<a href="https://colab.research.google.com/github/Rahul19873/ML-and-Deep-learning-project/blob/main/SMS_Classification_project_using_Naive_Bayes_and_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# import library

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pickle
nltk.download ('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [29]:
#upload dataset

from google.colab import files
uploaded=files.upload()


Saving spam.csv to spam (1).csv


In [30]:
#load dataset
import pandas as pd
df=pd.read_csv('spam.csv',encoding='latin-1')
df=df[['v1','v2']]
df.columns=['label','message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#data set information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [ ]:
df['label'].value_counts()

,count
label,
ham,4825
spam,747


In [31]:
# convert label
df['label']=df['label'].map({'ham':0,'spam':1})
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [32]:
Stemmer=PorterStemmer()
stop_words=set(stopwords.words('english'))
def clean_text(text):
  text = text.lower()
  # Allow spaces, digits, and some punctuation in addition to letters
  text=re.sub('[^a-zA-Z0-9 ]','',text)
  words=text.split()
  words=[
      Stemmer.stem(word) for word in words if word not in stop_words]

  return " " .join(words)

  # apply cleaning

df['message']=df['message'].apply(clean_text)
df.head()

,label,message
0,0,go jurong point crazi avail bugi n great world...
1,0,ok lar joke wif u oni
2,1,free entri 2 wkli comp win fa cup final tkt 21...
3,0,u dun say earli hor u c alreadi say
4,0,nah dont think goe usf live around though


In [33]:
# apply TF-IDF
vectorizer=TfidfVectorizer(max_features=5000)
X=vectorizer.fit_transform(df['message'])
y=df['label']

In [34]:
#Train_test_split
X_train,X_test,y_train,y_test,=train_test_split(X,y,test_size=0.2,random_state=42)

In [35]:
# Train Naive Bayes
from sklearn.naive_bayes import MultinomialNB as MultinomealNB
nb = MultinomealNB()
nb.fit(X_train,y_train)
nb_pred=nb.predict(X_test)

In [36]:
# Evaluate Naive Bayes
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print(accuracy_score(y_test,nb_pred))
print(classification_report(y_test,nb_pred))
print(confusion_matrix(y_test,nb_pred))

0.9739910313901345
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.81      0.89       150

    accuracy                           0.97      1115
   macro avg       0.99      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115

[[965   0]
 [ 29 121]]


In [37]:
# Train SVM
from sklearn.svm import SVC,LinearSVC
svm=SVC()
svm=LinearSVC()
svm.fit(X_train,y_train)
svm_pred=svm.predict(X_test)

In [38]:
# evaluate SVM
print('Accuracy')
print(accuracy_score(y_test,svm_pred))
print(classification_report(y_test,svm_pred))
print(confusion_matrix(y_test,svm_pred))

Accuracy
0.9811659192825112
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.98      0.87      0.93       150

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

[[963   2]
 [ 19 131]]


In [39]:
# compare accuracy
nb_accuracy= accuracy_score(y_test,nb_pred)
svm_accuracy=accuracy_score(y_test,svm_pred)
print(f'Naive Bayes Accuracy:{nb_accuracy}')
print(f'SVM:{svm_accuracy})')

Naive Bayes Accuracy:0.9739910313901345
SVM:0.9811659192825112)


In [40]:
# save the model
pickle.dump(
    nb,open("spam_model.pkl",'wb')
)
pickle.dump(vectorizer,open('tfidk.pkl','wb'))

In [44]:
# predict the new sms
def predict_sms(message):
  cleaned=clean_text(message)
  vector=vectorizer.transform([cleaned])
  prediction=svm.predict(vector)
  if prediction[0]==1:
    print('spam message')
  else:
      print('ham message')

In [45]:
predict_sms("Hi Rahul, are we meeting tomorrow at 10 AM?")

ham message


In [46]:
predict_sms('Congratulation! you win a free iphone.click here')

spam message
